In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行

env= 'local'
env= 'colab'
# env= 'kaggle'

# 核心参数
quick_test = True
random_seed = 42


In [2]:

if env == 'kaggle':
    # This Python 3 environment comes with many helpful analytics libraries installed
    # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
    # For example, here's several helpful packages to load

    # Input data files are available in the read-only "../input/" directory
    # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))

    micro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/micro_data.parquet')
    macro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/macro_data.parquet')
    # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
    # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

elif env == 'colab':
    # Colab 版本的代码
    # 1. 设置环境变量
    import google.colab 
    os.environ['KAGGLE_USERNAME'] = 'jianbinchenuc'
    os.environ['KAGGLE_KEY'] = 'fa590c922d3624b8e0734dc7a90dd95a'

    # 2. 安装 kaggle 库 (如果尚未安装)
    !pip install -q kaggle

    # 3. 创建目录并下载数据集
    # -p 参数指定下载目录，--unzip 参数会自动解压
    !mkdir -p /content/mdndata
    !kaggle datasets download -d jianbinchenuc/mdndata-1990 -p /content/mdndata --unzip

    # 4. 验证文件是否已存在
    print("已下载的文件：", os.listdir('/content/mdndata'))
    micro_df = pd.read_parquet('/content/mdndata/micro_data.parquet') # 可以根据需要切换到 Colab 的路径
    macro_df = pd.read_parquet('/content/mdndata/macro_data.parquet') # 可以根据需要切换到 Colab 的路径

elif env == 'local':
    micro_df = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/micro_data.parquet') # local 路径
    macro_df = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/macro_data.parquet') # local 路径

else:
    raise ValueError("Invalid environment specified. Please choose 'kaggle', 'colab', or 'local'.")

Dataset URL: https://www.kaggle.com/datasets/jianbinchenuc/mdndata-1990
License(s): unknown
100% 230M/230M [00:02<00:00, 95.7MB/s] 

已下载的文件： ['micro_data.parquet', 'macro_data.parquet']


In [3]:
import pandas as pd
import numpy as np
import sqlite3
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json
import copy
import warnings
import random
import time
from datetime import datetime
warnings.filterwarnings('ignore')
from scipy.stats import t      # 🌟 换成 t 分布
from scipy.optimize import brentq
from tqdm import tqdm
from scipy.stats import norm

# ==========================================
# 全局随机种子锁死函数
# ==========================================
def seed_everything(seed=42):
    """
    全局锁死随机种子，确保深度学习模型的绝对可复现性
    """
    # 1. 锁死 Python 内置随机库
    random.seed(seed)
    
    # 2. 锁死 Python 哈希表的随机化 (保证字典遍历顺序一致)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 3. 锁死 Numpy 随机种子
    np.random.seed(seed)
    
    # 4. 锁死 PyTorch CPU 随机种子
    torch.manual_seed(seed)
    
    # 5. 锁死 PyTorch GPU 随机种子
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # 如果你有双卡/多卡
        
        # 【极其关键】锁死 CuDNN 底层算法的随机性
        # 注意：设置为 True 可能会略微牺牲一点点 GPU 训练速度，但为了发论文复现，这是必须的！
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    print(f"Global seed set to {seed} to ensure reproducibility.")

# ==========================================
# 0. 自动设备检测
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ==========================================
# 1. 外置数据预处理模块 (ETL & Feature Engineering)
# ==========================================
print(">>> [1/3] Loading and Preprocessing Data (Externalized)...")

# micro_cols 是微观特征列，macro_cols 是宏观特征列，后续我们会对它们分别进行不同的预处理
micro_cols=micro_df.columns.drop(['mthcaldt', 'permno', 'mthret_lead1'])
macro_cols=macro_df.columns.drop(['date'])

# 1.1 日期对齐与合并
micro_df['mthcaldt'] = pd.to_datetime(micro_df['mthcaldt']) + pd.offsets.MonthEnd(0)
macro_df['date'] = pd.to_datetime(macro_df['date']) + pd.offsets.MonthEnd(0)

df = pd.merge(micro_df, macro_df, left_on='mthcaldt', right_on='date', how='left')
df = df.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)
df = df[(df['mthcaldt'] <= pd.to_datetime('2023-12-31')) & (df['mthcaldt'] >= pd.to_datetime('1990-01-31'))]  # 日期在1990-01-31到2023-12-31之间

# 1.2 目标收益率设定 (绝对禁止缩尾！)
df['target_ret_final'] = df['mthret_lead1']

# 1.3 缺失值预填充 (可在此处放置你的业务填充逻辑) 
# 可以考虑先做1.4再做1.3, 但是我觉得区别不大
# 例如：使用截面均值填充微观特征缺失值
for feature in micro_cols:
    if feature in df.columns:
        df[feature] = df[feature].fillna(df.groupby('mthcaldt')[feature].transform('mean'))

# 1.4 微观特征：截面秩标准化 (Cross-Sectional Rank Normalization) [-1, 1], 这个未来也可以考虑别的归一化的方法
print(" -> Applying Rank Normalization to Micro Features...")
micro_tensor_cols = []

# rank transformation
for col in micro_cols:
    norm_col = f'{col}_norm'
    # 使用 pct=True 转化为 0~1，再映射到 -1~1
    df[norm_col] = df.groupby('mthcaldt')[col].transform(
        lambda x: (x.rank(pct=True) * 2) - 1
    ).fillna(0)  # 如果全截面缺失，填0(中性)
    micro_tensor_cols.append(norm_col)

# or normalization
# for col in micro_cols:
#     norm_col = f'{col}_norm'
#     df[norm_col] = (df[col] - df.groupby('mthcaldt')[col].transform('mean')) / (
#         df.groupby('mthcaldt')[col].transform('std') + 1e-8)
#     df[norm_col] = df[norm_col].fillna(0)  # 如果全截面缺失，填0(中性)
#     micro_tensor_cols.append(norm_col)

# 1.5 宏观特征：时间序列 Z-score 标准化
print(" -> Applying Z-score Normalization to Macro Features...")
# macro_tensor_cols = []
# for col in macro_cols:
#     norm_col = f'{col}_norm'
#     df[norm_col] = (df[col] - df[col].mean()) / (df[col].std() + 1e-8)
#     df[norm_col] = df[norm_col].fillna(0)
#     macro_tensor_cols.append(norm_col)

# 在下面Expanding Window 逐窗口计算均值和标准差，保证每个月的宏观特征只使用当月及之前的数据进行标准化，完全避免未来数据泄漏
macro_tensor_cols = list(macro_cols)

# 1.6 清理最终的空值行并固化数据集
processed_df = df.dropna(subset=['target_ret_final'] + micro_tensor_cols + macro_tensor_cols).copy()
print(f"Data preprocessing complete! Final shape: {processed_df.shape}")
print(f"Micro Features: {len(micro_tensor_cols)}, Macro Features: {len(macro_tensor_cols)}")

# # (可选) 你可以在这里将 processed_df 存为 Parquet:
# processed_df.to_parquet('./data/fully_processed_model_input.parquet')
pd.concat([processed_df.iloc[:5], processed_df.iloc[-5:]])

Using device: cpu
>>> [1/3] Loading and Preprocessing Data (Externalized)...
 -> Applying Rank Normalization to Micro Features...
 -> Applying Z-score Normalization to Macro Features...
Data preprocessing complete! Final shape: (1613878, 98)
Micro Features: 33, Macro Features: 27


,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,bm,evm,pcf,divyield,gprof,npm,accrual,cfm,at_turn,rd_sale,inv_turn,debt_at,intcov_ratio,short_debt,curr_ratio,cash_ratio,ps,opmad,de_ratio,mktcap,capei,debt_ebitda,ocf_lct,gpm,roe,mthret_lead1,turnover_1m,mthcap_log,MOM12m,date,ret,tms,dfy,dfr,svar,vrp,lzrt,wtexas,skvw,tail,dtoy,dtoat,rdsp,avgcor,d/p,d/y,infl,ntis,ndrbl,disag,e/p,d/e,ygap,b/m,i/k,govik,crdstd,target_ret_final,mthret_norm,mthprc_norm,mthcap_norm,mthvol_norm,shrout_norm,bm_norm,evm_norm,pcf_norm,divyield_norm,gprof_norm,npm_norm,accrual_norm,cfm_norm,at_turn_norm,rd_sale_norm,inv_turn_norm,debt_at_norm,intcov_ratio_norm,short_debt_norm,curr_ratio_norm,cash_ratio_norm,ps_norm,opmad_norm,de_ratio_norm,mktcap_norm,capei_norm,debt_ebitda_norm,ocf_lct_norm,gpm_norm,roe_norm,turnover_1m_norm,mthcap_log_norm,MOM12m_norm
0,10001,1990-01-31,-0.018519,9.9375,1.015613e+04,3.525500e+04,1022,0.917331,4.853678,7.769246,0.050633,0.164538,0.050236,-0.005309,0.081263,1.223659,0.000000,91.308370,0.426239,3.131479,0.107954,1.173138,0.271814,0.421441,0.103437,2.235379,10.092250,16.372132,2.590528,0.339098,0.134464,0.151031,-0.006289,34.496086,9.225833,0.033145,1990-01-31,-0.067661,0.0101,0.0095,0.015200,0.002892,35.9054,-1.672983,0.040330,0.049831,0.465337,0.921851,0.921851,0.022034,0.307570,0.031282,0.031952,0.001589,-0.012334,0.065133,2.941912,0.067850,0.453103,-2.769167,0.399210,0.035913,0.034468,54.4,-0.006289,0.314896,0.223887,-0.429705,-0.699868,-0.952402,0.506831,-0.261789,0.353460,0.888938,-0.518290,0.403702,0.366681,0.275452,-0.085941,-0.335390,0.955928,0.593654,0.055972,-0.467166,-0.622741,-0.039665,-0.345527,0.496695,0.591891,-0.468488,0.107536,0.432349,0.288233,-0.539445,0.317320,-0.007052,-0.429705,0.000220
1,10001,1990-02-28,-0.006289,9.8750,1.009225e+04,1.486200e+04,1022,0.849363,4.626443,5.646409,0.050000,0.181426,0.052654,-0.024148,0.084097,1.301493,0.000000,96.509413,0.413490,3.119863,0.088959,1.243238,0.273871,0.403697,0.107955,2.084969,10.220000,15.386932,2.279116,0.508677,0.139398,0.162685,0.012821,14.542074,9.219523,0.014726,1990-02-28,0.013381,0.0102,0.0092,0.001300,0.001009,32.9177,-1.558446,-0.051101,0.058325,0.476048,0.934915,0.934915,0.018495,0.316860,0.033860,0.031530,0.010309,-0.013897,0.027459,2.929758,0.068800,0.462961,-2.753609,0.406415,0.035913,0.034468,54.4,0.012821,-0.234451,0.212836,-0.444640,-0.771063,-0.951478,0.389060,-0.326423,0.140273,0.889502,-0.483458,0.442876,0.226290,0.307896,0.005734,-0.417733,0.960300,0.546096,0.088663,-0.522276,-0.541685,-0.018968,-0.370093,0.533745,0.531539,-0.479929,0.075430,0.346273,0.473754,-0.561535,0.352448,-0.359065,-0.444640,-0.000221
2,10001,1990-03-31,0.012821,9.8750,1.014163e+04,1.272700e+04,1027,0.849363,4.626443,5.744959,0.049383,0.181426,0.052654,-0.024148,0.084097,1.301493,0.000000,96.509413,0.413490,3.119863,0.088959,1.243238,0.273871,0.410743,0.107955,2.084969,10.398375,15.655488,2.279116,0.508677,0.139398,0.162685,0.000000,12.392405,9.224404,-0.060729,1990-03-31,0.026588,0.0099,0.0084,0.003300,0.001032,26.5978,-1.523794,-0.058496,0.009316,0.488723,0.963369,0.963369,0.023228,0.335480,0.033838,0.034126,0.004710,-0.011729,-0.003898,3.042521,0.066890,0.473052,-2.780460,0.397226,0.035913,0.034468,54.4,0.000000,-0.001102,0.192197,-0.446771,-0.830725,-0.953934,0.381089,-0.309676,-0.015208,0.890677,-0.486886,0.443685,0.230769,0.307031,-0.000661,-0.422305,0.959445,0.548600,0.087503,-0.526119,-0.544192,-0.027992,-0.376681,0.532290,0.538462,-0.481596,0.044302,0.356844,0.469694,-0.558739,0.353317,-0.519065,-0.446771,0.001984
3,10001,1990-04-30,0.000000,9.8750,1.014163e+04,1.664500e+04,1027,0.849363,4.626443,5.674033,0.050000,0.181426,0.052654,-0.024148,0.084097,1.301493,0.000000,96.509413,0.413490,3.119863,0.088959,1.243238,0.273871,0.405672,0.107955,2.084969,10.270000,15.462210,2.279116,0.508677,0.139398,0.162685,-0.012658,16.207400,9.224404,0.072811,1990-04-30,-0.024504,0.0147,0.0084,0.001100,0.000967,26.2753,-1.525570,-0.085306,0.007114,0.486923,0.945416,0.9

In [4]:
# ==========================================
# 2. 轻量级时间窗口生成器
# ==========================================
class DataWindowGenerator:
    """
    仅负责按时间切分数据，向模型输送 Train/Val/Test 窗口
    """
    def __init__(self, df, date_col='mthcaldt'):
        self.df = df
        self.date_col = date_col
        
    def expanding_window_generator(self, initial_train_years=20, val_years=2, test_years=1):
        dates = np.sort(self.df[self.date_col].unique())
        initial_train_months = initial_train_years * 12
        val_months = val_years * 12
        test_months = test_years * 12
        
        start_idx = 0
        current_split_idx = initial_train_months
        
        while current_split_idx + val_months < len(dates):
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            current_split_idx += test_months

# ==========================================
# 2. 模型定义：结构化 MDN (Mixture Density Network)
# ==========================================
class StructuredMDN(nn.Module):
    def __init__(self, macro_dim, micro_dim, micro_hidden_dim=128, macro_hidden_dim=64, num_components=3, dropout_rate=0.2):
        """
        升级版结构化 MDN：
        - micro_hidden_dim 提升至 128 (为未来扩展到 100+ 特征做准备)
        - macro_hidden_dim 提升至 64 (为未来扩展到 100+ 特征做准备)
        - 加入 Dropout 防止对某个单一因子过度依赖 (过拟合)
        """
        super(StructuredMDN, self).__init__()
        self.num_components = num_components
        
        # ==========================================
        # 通道 A: Macro Network (门控网络 -> 预测机制权重 pi)
        # 宏观数据通常维度低且稳定，不需要太深
        # ==========================================
        self.macro_net = nn.Sequential(
            nn.Linear(macro_dim, macro_hidden_dim),
            nn.LayerNorm(macro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate), # 随机屏蔽一部分宏观因子，防止死记硬背
            nn.Linear(macro_hidden_dim , num_components)
        )
        
        # ==========================================
        # 通道 B: Micro Network (专家网络 -> 预测 mu 和 sigma)
        # 微观数据维度高、噪音大，稍微加深一点，并强力正则化
        # ==========================================
        self.micro_extractor = nn.Sequential(
            nn.Linear(micro_dim, micro_hidden_dim),
            nn.BatchNorm1d(micro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(micro_hidden_dim, micro_hidden_dim // 2),
            nn.BatchNorm1d(micro_hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout_rate)
        )
        
        # 预测头 (Heads)
        self.mu_head = nn.Linear(micro_hidden_dim // 2, num_components)
        
        # Sigma 必须严格为正，使用 Softplus
        self.sigma_head = nn.Linear(micro_hidden_dim // 2, num_components)

    def forward(self, x_macro, x_micro):
        pi_logits = self.macro_net(x_macro)
        h_micro = self.micro_extractor(x_micro)
        
        mu = self.mu_head(h_micro)
        sigma = nn.functional.softplus(self.sigma_head(h_micro)) + 1e-6 # softplus=log(1+exp(x)), 可以确保输出严格为正，避免数值不稳定
        
        return pi_logits, mu, sigma
    
# ==========================================
# 4. 稳健的 NLL 损失函数 (支持时间衰减权重)
# ==========================================
def mdn_nll_loss_weighted(pi_logits, mu, sigma, target, weights):
    """
    weights: shape (batch_size, 1)，距离现在越近权重大
    """
    log_pi = torch.log_softmax(pi_logits, dim=-1)

    # Normal 分布的 log_prob
    normal_dist = torch.distributions.Normal(mu, sigma)
    log_normal = normal_dist.log_prob(target.expand_as(mu))
    log_mix = log_pi + log_normal
    
    # 每个样本基础的 NLL loss
    loss = -torch.logsumexp(log_mix, dim=-1) 
    
    # 乘以时间衰减权重后求平均
    # 确保 weights 的 shape 与 loss 对齐
    weighted_loss = loss * weights.squeeze()
    
    return weighted_loss.mean()

# ==========================================
# 5. 极速版双通道训练循环 (All-in-VRAM 加速)
# ==========================================
def train_structured_mdn(model, 
                         X_train_macro, X_train_micro, y_train, w_train, 
                         X_val_macro=None, X_val_micro=None, y_val=None, w_val=None,
                         epochs=500, batch_size=8192, lr=1e-3, patience=20): # 注意 batch_size 变了！
    if quick_test == True:
        epochs = 2
        batch_size = 1024
        patience = 1

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # # 🌟 极速黑客技巧：直接在放入 Dataset 前，把所有的矩阵一次性全部送到 GPU！
    # # 这样在后面的 for 循环中，完全没有 CPU 到 GPU 的传输延迟
    # tr_dataset = TensorDataset(
    #     torch.FloatTensor(X_train_macro).to(device), 
    #     torch.FloatTensor(X_train_micro).to(device), 
    #     torch.FloatTensor(y_train).unsqueeze(1).to(device),
    #     torch.FloatTensor(w_train).unsqueeze(1).to(device)
    # )
    
    # va_dataset = TensorDataset(
    #     torch.FloatTensor(X_val_macro).to(device), 
    #     torch.FloatTensor(X_val_micro).to(device), 
    #     torch.FloatTensor(y_val).unsqueeze(1).to(device),
    #     torch.FloatTensor(w_val).unsqueeze(1).to(device)
    # )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6,
    )
    # 极度清爽，直接将传入的 Tensor 打包，不需要任何格式转换和设备转移！
    tr_dataset = TensorDataset(X_train_macro, X_train_micro, y_train.unsqueeze(1), w_train.unsqueeze(1))
    va_dataset = TensorDataset(X_val_macro, X_val_micro, y_val.unsqueeze(1), w_val.unsqueeze(1))
    
    # 因为数据已经在显卡里了，不需要复杂的 worker
    train_loader = DataLoader(tr_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(va_dataset, batch_size=batch_size, shuffle=False)
    
    best_val_loss = float('inf')
    best_model_state = None
    epochs_no_improve = 0
    
    # 🌟 新增：用于记录 Loss 历史
    train_loss_history = []
    val_loss_history = []
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        # 注意看：这里去掉了 .to(device)！因为数据切片本来就在显存里了！
        for b_macro, b_micro, b_y, b_w in train_loader:
            optimizer.zero_grad()
            
            # pi_logits, mu, sigma = model(b_macro, b_micro)

            pi_logits, mu, sigma = model(b_macro, b_micro)
            loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, b_y, b_w)

            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * b_macro.size(0)
            
        train_loss /= len(train_loader.dataset)
        train_loss_history.append(train_loss)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for b_macro, b_micro, b_y, b_w in val_loader:
                pi_logits, mu, sigma = model(b_macro, b_micro)
                loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, b_y, b_w)
                val_loss += loss.item() * b_macro.size(0)
        val_loss /= len(val_loader.dataset)
        val_loss_history.append(val_loss)

        scheduler.step(val_loss)
            
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break
                
    model.load_state_dict(best_model_state)
    return model, best_val_loss, train_loss_history, val_loss_history

# ==========================================
# 6. 启动模型训练流水线
# ==========================================
def calculate_time_decay_weights(df_dates, half_life_months=36):
    """
    计算基于半衰期的指数衰减权重。
    参数：
    df_dates: pd.Series, 包含 mthcaldt 日期
    half_life_months: 半衰期(月)。默认 36 个月(3年)权重减半。
    """
    max_date = df_dates.max()
    # 计算每个样本距离该窗口最新日期的月数差
    months_diff = (max_date.year - df_dates.dt.year) * 12 + (max_date.month - df_dates.dt.month)
    
    # 衰减因子 lambda: (0.5)^(1/half_life)
    decay_factor = np.power(0.5, 1.0 / half_life_months)
    
    # 计算基础权重
    weights = np.power(decay_factor, months_diff).values
    
    # 【极度关键】归一化权重，使得权重的均值为 1。
    # 如果不归一化，整个 Loss 的绝对值会变小，导致梯度的步长变相缩小！
    weights = weights / weights.mean() 
    return weights

In [5]:
if __name__ == "__main__":
    seed_everything(seed=random_seed)
    print("\n>>> [2/3] Initializing Data Window Generator and Global VRAM Tensors...")
        
    # 1. 实例化数据窗口生成器 (由于我们要用 Index 切片，重置一下 df 的索引非常重要)
    processed_df = processed_df.reset_index(drop=True)
    generator = DataWindowGenerator(df=processed_df, date_col='mthcaldt')
    window_gen = generator.expanding_window_generator(initial_train_years=15, val_years=1, test_years=1)

    # ==============================================================
    # 🌟 终极黑客技巧：全局数据一次性打入 GPU (Global All-in-VRAM)
    # ==============================================================
    print("    -> Pushing the entire 30-year dataset into GPU VRAM at once...")
    global_X_mac = torch.FloatTensor(processed_df[macro_tensor_cols].values).to(device)
    global_X_mic = torch.FloatTensor(processed_df[micro_tensor_cols].values).to(device)
    global_Y     = torch.FloatTensor(processed_df['target_ret_final'].values).to(device)
    # ==============================================================

    history_logs = []
    all_oos_predictions = []
    best_k = 5
    macro_dim = len(macro_tensor_cols)
    micro_dim = len(micro_tensor_cols)
    global_model = StructuredMDN(macro_dim, micro_dim, num_components=best_k).to(device)

    print("\n>>> [3/3] Starting Expanding Window Structured MDN Training (Zero PCIe Overhead)...")
    for window_idx, (info, train_df, val_df, test_df) in enumerate(window_gen):
        print(f"\n[Window {window_idx + 1}] Train: {info['train'][0]}~{info['train'][1]} | Val: {info['val'][0]}~{info['val'][1]} | Test: {info['test'][0]}~{info['test'][1]}")
        # --- 记录开始时间 ---
        start_time = time.time()

        # 提取各个集合在全局大表中的行索引 (Index)
        idx_tr = train_df.index.values
        idx_va = val_df.index.values
        idx_te = test_df.index.values

        # 先拿到 train 月份的唯一宏观值（时间等权）
        train_months = train_df['mthcaldt'].unique()
        mac_for_norm = macro_df[macro_df['date'].isin(train_months)][macro_tensor_cols]
        mac_mean = torch.FloatTensor(mac_for_norm.mean().values).to(device)
        mac_std  = torch.FloatTensor(mac_for_norm.std().values).to(device) + 1e-8

        # 再应用到 merged 的 global tensor
        X_tr_mac = (global_X_mac[idx_tr] - mac_mean) / mac_std
        X_va_mac = (global_X_mac[idx_va] - mac_mean) / mac_std
        X_te_mac = (global_X_mac[idx_te] - mac_mean) / mac_std

        X_tr_mic, Y_tr = global_X_mic[idx_tr], global_Y[idx_tr]
        X_va_mic, Y_va = global_X_mic[idx_va], global_Y[idx_va]
        X_te_mic       = global_X_mic[idx_te]

        # # --- 显存内部极速切片 (零搬运延迟) ---
        # X_tr_mac, X_tr_mic, Y_tr = global_X_mac[idx_tr], global_X_mic[idx_tr], global_Y[idx_tr]
        # X_va_mac, X_va_mic, Y_va = global_X_mac[idx_va], global_X_mic[idx_va], global_Y[idx_va]
        # X_te_mac, X_te_mic       = global_X_mac[idx_te], global_X_mic[idx_te]
        
        # --- 唯一需要传输的是这一轮的动态权重 (体积极小，几乎0耗时) ---
        w_train_np = calculate_time_decay_weights(train_df['mthcaldt'], half_life_months=36)
        w_val_np   = calculate_time_decay_weights(val_df['mthcaldt'], half_life_months=12)
        w_train = torch.FloatTensor(w_train_np).to(device)
        w_val   = torch.FloatTensor(w_val_np).to(device)
        
        # 动态学习率
        current_lr = 1e-3 if window_idx == 0 else 3e-4
        
        # 训练模型 (此时传入的全是已经在显存里的 Tensor，速度起飞)
        global_model, best_v_loss, tr_hist, va_hist = train_structured_mdn(
            model=global_model,     
            X_train_macro=X_tr_mac, X_train_micro=X_tr_mic, y_train=Y_tr, w_train=w_train,
            X_val_macro=X_va_mac, X_val_micro=X_va_mic, y_val=Y_va, w_val=w_val,
            lr=current_lr , epochs=1
        )

        # ----------------------------------------------------
        # 基于 Train + Val 联合集合进行 Fine-tune (防止灾难性遗忘)
        # ----------------------------------------------------
        best_epoch = len(tr_hist)
        finetune_epochs = max(5, best_epoch // 10)  # 只用10%的epoch做微调
        print(f"  Phase 1 done: best_epoch={best_epoch}, finetune_epochs={finetune_epochs}")
        
        # 🌟 修复：合并 Train 和 Val 的 Tensor，保持历史记忆的同时吃进最新数据
        X_tr_mac_ft = torch.cat([X_tr_mac, X_va_mac], dim=0)
        X_tr_mic_ft = torch.cat([X_tr_mic, X_va_mic], dim=0)
        Y_tr_ft     = torch.cat([Y_tr, Y_va], dim=0)
        
        # 重新计算合并后的时间衰减权重 (严谨做法)
        train_val_df = pd.concat([train_df, val_df])
        w_ft_np = calculate_time_decay_weights(train_val_df['mthcaldt'], half_life_months=36)
        w_ft = torch.FloatTensor(w_ft_np).to(device)

        global_model, _, _, _ = train_structured_mdn(
            model=global_model,
            X_train_macro=X_tr_mac_ft, X_train_micro=X_tr_mic_ft, # 使用联合数据
            y_train=Y_tr_ft,           w_train=w_ft,
            X_val_macro=X_va_mac,      X_val_micro=X_va_mic,      # 仅作占位，无需关心
            y_val=Y_va,                w_val=w_val,
            lr=current_lr * 0.1,       # 学习率给 0.1 即可，0.05 也行，平滑过渡
            epochs=finetune_epochs,
            patience=finetune_epochs + 1  # 强制跑完，禁用 early stop
        )
        print(f"  Phase 2 fine-tune done ({finetune_epochs} epochs on Train+Val)")


        # --- 记录结束时间 ---
        end_time = time.time()
        duration = end_time - start_time
        # --- 记录指标 ---
        history_logs.append({
            'window': window_idx + 1,
            'train_period': f"{info['train'][0]} to {info['train'][1]}",
            'test_period': f"{info['test'][0]} to {info['test'][1]}",
            'duration_sec': duration,
            'best_val_loss': best_v_loss,
            'epochs_run': len(tr_hist),
            'train_loss_trace': tr_hist, # 存储为一个 list，方便后续绘图
            'val_loss_trace': va_hist
        })
        
        # --- 样本外预测 ---
        global_model.eval()
        with torch.no_grad():
            pi_logits, mu, sigma = global_model(X_te_mac, X_te_mic)
            pi = torch.softmax(pi_logits, dim=-1)
            
        # --- 结果回传至 CPU 保存 ---
        preds_df = test_df[['permno', 'mthcaldt', 'target_ret_final']].copy()
        preds_df['pi_vec'] = [json.dumps(vec.tolist()) for vec in pi.cpu().numpy()]
        preds_df['pi_logits_vec'] = [json.dumps(vec.tolist()) for vec in pi_logits.cpu().numpy()]  # 新增
        preds_df['mu_vec'] = [json.dumps(vec.tolist()) for vec in mu.cpu().numpy()]
        preds_df['sigma_vec'] = [json.dumps(vec.tolist()) for vec in sigma.cpu().numpy()]
        # preds_df['nu_vec']        = [json.dumps(vec.tolist()) for vec in nu.cpu().numpy()]
        all_oos_predictions.append(preds_df)
        
        # 测试用，仅跑几个窗口
        if quick_test == True and window_idx == 1: break

    print("\n>>> Pipeline complete! Aggregating all Out-of-Sample predictions...")
    if len(all_oos_predictions) > 0:
        final_oos_df = pd.concat(all_oos_predictions, ignore_index=True)
        metrics_df = pd.DataFrame(history_logs)
        print(f"Successfully generated {len(final_oos_df)} predictions.")

Global seed set to 42 to ensure reproducibility.

>>> [2/3] Initializing Data Window Generator and Global VRAM Tensors...
    -> Pushing the entire 30-year dataset into GPU VRAM at once...

>>> [3/3] Starting Expanding Window Structured MDN Training (Zero PCIe Overhead)...

[Window 1] Train: 1990-01~2004-12 | Val: 2005-01~2005-12 | Test: 2006-01~2006-12
  Phase 1 done: best_epoch=2, finetune_epochs=5
  Phase 2 fine-tune done (5 epochs on Train+Val)

[Window 2] Train: 1990-01~2005-12 | Val: 2006-01~2006-12 | Test: 2007-01~2007-12
  Phase 1 done: best_epoch=2, finetune_epochs=5
  Phase 2 fine-tune done (5 epochs on Train+Val)

>>> Pipeline complete! Aggregating all Out-of-Sample predictions...
Successfully generated 88593 predictions.


In [6]:
import pandas as pd
import numpy as np
import sqlite3
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json
import copy
import warnings
import random
import time
from datetime import datetime
warnings.filterwarnings('ignore')
from scipy.stats import t      # 🌟 换成 t 分布
from scipy.optimize import brentq
from tqdm import tqdm
from scipy.stats import norm

pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行

# 如果没有final_oos_df, 则从parquet加载
if 'final_oos_df' not in locals():
    print(">>> Loading final_oos_df from Parquet...")
    final_oos_df = pd.read_parquet('/Users/jianbinchen/Downloads/final_oos_df_20260430_0833.parquet') # local 路径
    print(f"final_oos_df loaded with shape: {final_oos_df.shape}")

# =================================================================
# 1. 核心数学工具：混合高斯分布的 CDF 和 ES 计算
# =================================================================

def mdn_cdf(y, pi, mu, sigma):
    """
    计算混合高斯分布在 y 处的累积概率 P(Y <= y)
    公式: F(y) = sum( pi_k * Phi((y - mu_k) / sigma_k) )
    """
    # norm.cdf 是标准正态分布的累积函数
    return np.sum(pi * norm.cdf((y - mu) / sigma))

def mdn_var_objective(y, pi, mu, sigma, alpha):
    """
    求解 VaR 的目标函数：F(y) - alpha = 0
    """
    return mdn_cdf(y, pi, mu, sigma) - alpha

def calculate_es_analytical(alpha, pi, mu, sigma):
    """
    混合高斯分布的 Expected Shortfall (ES) 解析解
    公式参考: ES_alpha = -(1/alpha) * sum( pi_k * mu_k * Phi(d1) - sigma_k * phi(d1) )
    其中 d1 是标准化后的 VaR 阈值
    """
    # 1. 首先通过数值求根找到 VaR (使得累积概率等于 alpha)
    # 搜索区间设定在均值的上下 10 倍标准差之间，确保覆盖尾部
    try:
        lower_bound = np.min(mu - 10 * sigma)
        upper_bound = np.max(mu + 10 * sigma)
        var_alpha = brentq(mdn_var_objective, lower_bound, upper_bound, args=(pi, mu, sigma, alpha))
    except:
        # 如果求根失败（极罕见），使用加权均值作为退路
        print(">>> Warning: Root finding for VaR failed. Using weighted mean as fallback.")
        return np.sum(pi * mu), np.sum(pi * mu)

    # 2. 计算每个组件在 VaR 处的贡献
    # d = (VaR - mu) / sigma
    d = (var_alpha - mu) / sigma
    
    # phi 是标准正态概率密度函数，Phi 是标准正态累积分布函数
    term1 = mu * norm.cdf(d)
    term2 = sigma * norm.pdf(d)
    
    # ES = -(1/alpha) * Integral from -inf to VaR of (y * f(y)) dy
    # 对于混合高斯，积分具有解析形式
    es_val = -(1/alpha) * np.sum(pi * (term1 - term2))
    
    return var_alpha, es_val

def calculate_crps_numerical(y_true, pi, mu, sigma):
    """
    通过数值积分计算混合高斯分布的 CRPS
    """
    # 1. 构建积分网格 z
    # 因为我们的目标收益率已经用 VIX 缩放过，绝大多数值落在 [-5, 5] 之间。
    # 设置 [-10, 10] 的区间和 1000 个格点，足以保证极高的积分精度。
    z = np.linspace(-10, 10, 1000)
    
    # 2. 计算网格上每一个点的预测累积概率 F(z)
    cdf_z = np.zeros_like(z)
    for k in range(len(pi)):
        # 向量化计算：算出每一个网格点的正态 CDF 并加权
        cdf_z += pi[k] * norm.cdf((z - mu[k]) / sigma[k])
        
    # 3. 真实发生的阶跃函数 (Heaviside step function)
    # 当 z 小于真实收益率时为 0，大于等于时为 1
    step_z = (z >= y_true).astype(float)
    
    # 4. 计算平方差
    squared_diff = (cdf_z - step_z) ** 2
    
    # 5. 使用梯形法则 (Trapezoidal rule) 计算定积分
    crps_val = np.trapezoid(squared_diff, z)
    
    return crps_val

# =================================================================
# 2. 主程序：从 SQLite 读取数据并执行计算
# =================================================================

results = []

print(">>> Extracting Tail Risk Factors (ES & VaR)...")
# 使用 tqdm 显示进度
for idx, row in tqdm(final_oos_df.iterrows(), total=len(final_oos_df)):
    # 1. 解码 JSON 字符串回 Numpy 数组
    pi = np.array(json.loads(row['pi_vec']))
    mu = np.array(json.loads(row['mu_vec']))
    sigma = np.array(json.loads(row['sigma_vec']))
    y_true = row['target_ret_final'] # 真实发生的（缩放后的）收益率
    
    # 2. 计算风险指标：VaR 和 ES
    # 计算 1% 分位数的 VaR 和 ES
    alpha = 0.01
    var_1, es_1 = calculate_es_analytical(alpha, pi, mu, sigma)
    #计算 5% 分位数的 VaR 和 ES
    alpha = 0.05
    var_5, es_5 = calculate_es_analytical(alpha, pi, mu, sigma)
    # 计算 10% 分位数的 VaR 和 ES
    alpha = 0.10
    var_10, es_10 = calculate_es_analytical(alpha, pi, mu, sigma)

    # 3. 计算 PIT (Probability Integral Transform)
    # 即：真实收益率 y_true 在预测分布中处于什么百分位
    pit_val = mdn_cdf(y_true, pi, mu, sigma)

    # 新增：计算 CRPS
    crps_val = calculate_crps_numerical(y_true, pi, mu, sigma)

    # 4. 计算分布的高阶矩 (用于控制变量)
    # 混合分布均值: sum(pi * mu)
    mean_pred = np.sum(pi * mu)
    # 混合分布方差: sum(pi * (mu^2 + sigma^2)) - mean^2
    var_pred = np.sum(pi * (mu**2 + sigma**2)) - mean_pred**2
    
    results.append({
        'permno': row['permno'],
        'date': row['mthcaldt'],
        'es_1': es_1,
        'var_1': var_1,
        'es_5': es_5,       
        'var_5': var_5,     
        'es_10': es_10,     
        'var_10': var_10,     
        'pit': pit_val,     
        'crps': crps_val,   # <--- 把 CRPS 也存进最终的 DataFrame 里
        'mean_pred': mean_pred,
        'vol_pred': np.sqrt(var_pred),
        'realized_ret': y_true
    })

ES_df = pd.DataFrame(results)

>>> Extracting Tail Risk Factors (ES & VaR)...


100%|██████████| 88593/88593 [10:03<00:00, 146.88it/s]


In [7]:
# 指定保存路径
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

if env == 'kaggle':
    path = '/kaggle/working'
elif env == 'colab':
    # 1. 挂载 Google Drive
    google.colab.drive.mount('/content/drive')
    path = '/content/drive/MyDrive/Colab Notebooks/'
else:
    path = '/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output'

# 执行保存
final_oos_df.to_parquet(os.path.join(path, f'final_oos_df_{timestamp}.parquet'), index=False)
metrics_df.to_parquet(os.path.join(path, f'metrics_df_{timestamp}.parquet'), index=False)
ES_df.to_parquet(os.path.join(path, f'ES_df_{timestamp}.parquet'), index=False)
print(f"保存成功！文件位置: {path}")


Mounted at /content/drive
保存成功！文件位置: /content/drive/MyDrive/Colab Notebooks/


In [8]:
if env == 'colab':
    # 运行完你的所有逻辑后，调用此命令会自动释放当前后端实例
    google.colab.runtime.unassign()